# RL-2026s1-tp — 통신 손실 강건성 Sweep (Colab + Google Drive)

학습된 정책 A(베이스라인) vs 정책 B(통신 손실 [0, 0.1] DR로 학습)를 6개 손실률에서 비교해 강건성 곡선을 만든다.

## 사전 준비
Drive에 아래 구조로 파일 업로드 완료해야 함.

```
{WORK_DIR}/
├── comm_env.py
├── formation_seq.py
├── comm_eval.py
├── comm_fail_sweep.py
└── RL/RL/
    ├── ckpt_dg_comm_base/ckpt_300.pt
    └── ckpt_dg_comm_robust/ckpt_280.pt
```

## 런타임
CPU로 충분 (상단 메뉴: 런타임 → 런타임 유형 변경 → CPU). Sweep 자체가 평가만이라 GPU도 큰 이득 없음.

## 1. Google Drive 마운트 + 파일 구조 점검

`WORK_DIR`만 본인이 업로드한 폴더로 바꾸면 다음 셀들은 그대로 실행됩니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ☝️ 본인이 업로드한 Drive 폴더로 바꿔주세요
WORK_DIR = "/content/drive/MyDrive/RL-sweep"

%cd {WORK_DIR}
print("\n=== 폴더 내용 ===")
!ls -la
print("\n=== A ckpt 폴더 (앞 5개) ===")
!ls RL/RL/ckpt_dg_comm_base/ | head -5
print("\n=== B ckpt 폴더 (앞 5개) ===")
!ls RL/RL/ckpt_dg_comm_robust/ | head -5

## 2. 패키지 설치 (한 번만)

torchrl/tensordict는 버전을 고정합니다. 안 고정하면 최신 torchrl이 깔려 ckpt 로드 시 API 차이로 깨질 수 있어요.

In [ ]:
!pip install -q torchrl==0.12.0 tensordict==0.12.4 pettingzoo==1.24.3 gymnasium scipy matplotlib pillow tensorboard

## 3. Sweep 실행 (ckpt_300 vs ckpt_280)

결과 PNG와 log가 Drive에 직저장됩니다 → 런타임 끊겨도 살아남음.

- 1200 에피소드 (6 손실률 × 2 정책 × 100 ep)
- CPU에서 5~15분 소요
- `tee` 대신 `>` 로 redirect → tqdm 진행바가 log에 안 섞임

In [ ]:
%cd {WORK_DIR}

!python comm_fail_sweep.py \
  --ckpt-a RL/RL/ckpt_dg_comm_base/ckpt_300.pt \
  --ckpt-b RL/RL/ckpt_dg_comm_robust/ckpt_280.pt \
  --shapes "GROUND,D,G" --max-steps 320 \
  --comm-fail-levels 0.0,0.05,0.1,0.15,0.2,0.3 \
  --n-episodes 100 --device cpu --greedy \
  --out comm_fail_sweep_dg_fixed.png \
  > sweep_fixed.log 2>&1

print("=== sweep 끝, 결과 표 ===")
!cat sweep_fixed.log

## 4. 결과 그림 표시

In [ ]:
from IPython.display import Image
Image(f"{WORK_DIR}/comm_fail_sweep_dg_fixed.png")

## 5. (선택) `best.pt`로 한 번 더 — 안정 구간 최고 ckpt

`ckpt_dg_comm_base/best.pt`, `ckpt_dg_comm_robust/best.pt`가 있다면 이게 보통 더 좋은 결과를 줍니다.

In [ ]:
%cd {WORK_DIR}

!python comm_fail_sweep.py \
  --ckpt-a RL/RL/ckpt_dg_comm_base/best.pt \
  --ckpt-b RL/RL/ckpt_dg_comm_robust/best.pt \
  --shapes "GROUND,D,G" --max-steps 320 \
  --comm-fail-levels 0.0,0.05,0.1,0.15,0.2,0.3 \
  --n-episodes 100 --device cpu --greedy \
  --out comm_fail_sweep_dg_best.png \
  > sweep_best.log 2>&1

print("=== best.pt sweep 결과 ===")
!cat sweep_best.log

from IPython.display import Image
Image(f"{WORK_DIR}/comm_fail_sweep_dg_best.png")

## 6. (선택) 정책 동작 GIF로 시각화

Sweep 표만으로는 정책이 **왜** 그렇게 행동했는지 직관이 안 옵니다. 한 에피소드를 GIF로 뽑아서 실제 드론 움직임을 눈으로 확인.

세 가지 케이스를 따로 만듭니다:

| GIF | 정책 | comm_fail | 보고 싶은 것 |
|---|---|---|---|
| `demo_A_clean.gif` | A | 0.0 | 베이스라인이 어떻게 망가지는지 (멈춤? 헤맴?) |
| `demo_B_clean.gif` | B | 0.0 | "충돌 578회" 모드의 실체 — 정말 떼로 부딪히는지 |
| `demo_B_noisy.gif` | B | 0.3 | success 99% 시 정상 작동 모습 |

각 GIF 생성에 ~1분 소요 (CPU 기준). 셀 3개 = 약 3분.

In [ ]:
%cd {WORK_DIR}

# A 정책 — comm_fail=0 (베이스라인, 망가진 상태 확인)
!python comm_eval.py \
  --ckpt RL/RL/ckpt_dg_comm_base/ckpt_300.pt \
  --shapes "GROUND,D,G" --max-steps 320 \
  --greedy --n-episodes 10 \
  --comm-fail-prob 0.0 \
  --device cpu \
  --save-gif demo_A_clean.gif \
  --out eval_A_clean.txt

print("=== A 평가 결과 ===")
!cat eval_A_clean.txt

In [ ]:
%cd {WORK_DIR}

# B 정책 — comm_fail=0 (충돌 폭주 모드 시각 확인)
!python comm_eval.py \
  --ckpt RL/RL/ckpt_dg_comm_robust/ckpt_280.pt \
  --shapes "GROUND,D,G" --max-steps 320 \
  --greedy --n-episodes 10 \
  --comm-fail-prob 0.0 \
  --device cpu \
  --save-gif demo_B_clean.gif \
  --out eval_B_clean.txt

print("=== B at comm_fail=0 평가 결과 ===")
!cat eval_B_clean.txt

In [ ]:
%cd {WORK_DIR}

# B 정책 — comm_fail=0.3 (정상 작동 케이스 시각 확인)
!python comm_eval.py \
  --ckpt RL/RL/ckpt_dg_comm_robust/ckpt_280.pt \
  --shapes "GROUND,D,G" --max-steps 320 \
  --greedy --n-episodes 10 \
  --comm-fail-prob 0.3 \
  --device cpu \
  --save-gif demo_B_noisy.gif \
  --out eval_B_noisy.txt

print("=== B at comm_fail=0.3 평가 결과 ===")
!cat eval_B_noisy.txt

In [ ]:
from IPython.display import Image, display, Markdown

display(Markdown("### A 정책 — comm_fail = 0 (망가진 베이스라인)"))
display(Image(f"{WORK_DIR}/demo_A_clean.gif"))

display(Markdown("### B 정책 — comm_fail = 0 (충돌 폭주 가설 검증)"))
display(Image(f"{WORK_DIR}/demo_B_clean.gif"))

display(Markdown("### B 정책 — comm_fail = 0.3 (정상 작동)"))
display(Image(f"{WORK_DIR}/demo_B_noisy.gif"))

---

## 트러블슈팅

- **셀 1에서 `No such file or directory`** → `WORK_DIR` 경로가 틀림. Drive에서 실제 폴더 경로를 복사해 수정.
- **셀 3에서 `ModuleNotFoundError: torchrl`** → 셀 2가 안 돌았거나 런타임을 재시작함. 셀 2 다시 실행.
- **셀 3에서 `FileNotFoundError: ckpt_300.pt`** → ckpt 경로 확인. `!find RL -name 'ckpt_300*'` 같은 독립 셀로 실제 위치 찾기.
- **sweep 결과가 온통 0%** → 다른 ckpt로 다시 시도. 아래 ckpt들이 안전권:
  - A: `ckpt_280.pt`, `ckpt_300.pt`, `ckpt_400.pt`, `best.pt`
  - B: `ckpt_270.pt`, `ckpt_280.pt`, `best.pt`
- **런타임 끊김** → Drive에 결과가 이미 저장되어 있음. 셀 1+2 재실행 후 셀 4로 바로 PNG 보면 됨.